# Análisis de Ventas y Rendimiento - Aura Beauty Co.
**Autor:** Maria Rico 

**Herramientas:** Python (Pandas, NumPy), Power BI  

---

## 1. Contexto de Negocio
**Aura Beauty Co.** es una marca de productos cosméticos que vende a través de múltiples canales (E-commerce, Tiendas Físicas y Distribuidores). El equipo directivo necesita entender el rendimiento de sus productos y canales para optimizar las estrategias de comercialización y reducir el impacto financiero de las devoluciones.

### Pregunta Principal de Negocio
> **¿Cómo podemos optimizar el rendimiento de ventas y el margen de ganancia por canal y categoría de producto, identificando al mismo tiempo el impacto de las devoluciones en el margen neto?**

---

## 2. Generación del Dataset Sintético
En esta sección importamos las librerías necesarias e inicializamos la generación de $1{,}500$ registros de ventas sintéticos. 

Para simular un entorno de datos real, introduciremos deliberadamente **anomalías**:
* Valores nulos (`NaN`) en precios unitarios.
* Inconsistencias de formato en la columna de `Canal` (espacios extra y variaciones de caja).

In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Configuración de semilla para reproducibilidad
np.random.seed(42)

# --- 1. GENERACIÓN DEL DATASET SINTÉTICO ---
n_rows = 1500

fechas = [datetime(2025, 1, 1) + timedelta(days=int(x)) for x in np.random.randint(0, 365, n_rows)]
canales = np.random.choice(['E-commerce', 'Retail Store', 'Distributor'], size=n_rows, p=[0.5, 0.4, 0.1])
categorias = np.random.choice(['Skin Care', 'Makeup', 'Hair Care', 'Fragrance'], size=n_rows, p=[0.4, 0.3, 0.2, 0.1])

productos_dict = {
    'Skin Care': ['Sérum Vitamina C', 'Crema Hidratante', 'Limpiador Facial'],
    'Makeup': ['Base de Maquillaje', 'Labial Matte', 'Mascara de Pestañas'],
    'Hair Care': ['Shampoo Repaparador', 'Acondicionador', 'Aceite Capilar'],
    'Fragrance': ['Perfume Floral 50ml', 'Body Mist']
}

productos = [np.random.choice(productos_dict[cat]) for cat in categorias]
cantidades = np.random.randint(1, 6, size=n_rows)
precios_unitarios = np.random.uniform(12.0, 85.0, size=n_rows).round(2)
costos_unitarios = (precios_unitarios * np.random.uniform(0.3, 0.6, size=n_rows)).round(2)
devoluciones = np.random.choice([0, 1], size=n_rows, p=[0.92, 0.08])

df = pd.DataFrame({
    'ID_Venta': [f'ORD-{1000 + i}' for i in range(n_rows)],
    'Fecha': fechas,
    'Canal': canales,
    'Categoria': categorias,
    'Producto': productos,
    'Cantidad': cantidades,
    'Precio_Unitario': precios_unitarios,
    'Costo_Unitario': costos_unitarios,
    'Devuelto': devoluciones
})

# Introducir "ruido" a propósito para practicar la limpieza
df.loc[df.sample(frac=0.03).index, 'Precio_Unitario'] = np.nan  # Valores nulos
df.loc[df.sample(frac=0.02).index, 'Canal'] = '  e-commerce '    # Formatos inconsistentes

## 3. Exploración Inicial de Datos (EDA)
Antes de realizar cualquier transformación, inspeccionamos la estructura general de los datos, los tipos de variables y la presencia de valores faltantes.

In [6]:
print("--- Primeras filas del dataset original ---")
display(df.head())
df.info()

--- Primeras filas del dataset original ---


,ID_Venta,Fecha,Canal,Categoria,Producto,Cantidad,Precio_Unitario,Costo_Unitario,Devuelto
0,ORD-1000,2025-04-13,E-commerce,Makeup,Base de Maquillaje,3,22.63,12.48,0
1,ORD-1001,2025-12-15,Distributor,Makeup,Mascara de Pestañas,2,62.91,20.24,0
2,ORD-1002,2025-09-28,E-commerce,Makeup,Mascara de Pestañas,5,58.92,33.25,0
3,ORD-1003,2025-04-17,E-commerce,Skin Care,Sérum Vitamina C,3,69.14,26.07,1
4,ORD-1004,2025-03-13,E-commerce,Skin Care,Limpiador Facial,2,80.44,38.44,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   ID_Venta         1500 non-null   object        
 1   Fecha            1500 non-null   datetime64[ns]
 2   Canal            1500 non-null   object        
 3   Categoria        1500 non-null   object        
 4   Producto         1500 non-null   object        
 5   Cantidad         1500 non-null   int32         
 6   Precio_Unitario  1455 non-null   float64       
 7   Costo_Unitario   1500 non-null   float64       
 8   Devuelto         1500 non-null   int64         
dtypes: datetime64[ns](1), float64(2), int32(1), int64(1), object(4)
memory usage: 99.7+ KB


## 4. Limpieza de Datos y Transformación (ETL)

En este paso aplicamos las siguientes reglas de negocio y limpieza:

1. **Estandarización de Texto:** Eliminación de espacios en blanco y normalización de nombres en la columna `Canal`.
2. **Tratamiento de Nulos:** Imputación de precios unitarios faltantes utilizando la media según la categoría de producto.
3. **Cálculo de Métricas Financieras:**
   * $\text{Venta Total} = \text{Cantidad} \times \text{Precio Unitario}$
   * $\text{Ganancia Bruta} = \text{Venta Total} - \text{Costo Total}$
   * $\text{Venta Neta} = \text{Venta Total}$ (Ajustado a $0$ si el producto fue devuelto).

In [7]:
# --- 2. LIMPIEZA DE DATOS (ETL) ---

# a) Normalizar texto (eliminar espacios y corregir mayúsculas/minúsculas)
df['Canal'] = df['Canal'].astype(str).str.strip().str.title()
df['Canal'] = df['Canal'].replace({'E-Commerce': 'E-commerce'})

# b) Tratar valores nulos en Precio_Unitario imputando la media por producto
df['Precio_Unitario'] = df.groupby('Producto')['Precio_Unitario'].transform(lambda x: x.fillna(x.mean()))

# c) Crear métricas financieras calculadas
df['Venta_Total'] = (df['Cantidad'] * df['Precio_Unitario']).round(2)
df['Costo_Total'] = (df['Cantidad'] * df['Costo_Unitario']).round(2)
df['Ganancia_Bruta'] = df['Venta_Total'] - df['Costo_Total']
df['Margen_%'] = ((df['Ganancia_Bruta'] / df['Venta_Total']) * 100).round(2)

# d) Ajuste por devoluciones (si se devolvió, la venta neta ajustada es 0)
df['Venta_Neta'] = np.where(df['Devuelto'] == 1, 0, df['Venta_Total'])

## 5. Validación y Exportación
Verificamos que el dataset limpio no contenga valores nulos ni inconsistencias de formato antes de proceder con su exportación a un archivo `.csv` para su posterior consumo en **Power BI**.

In [8]:
print("\n--- Dataset Limpio y Preparado ---")
print(df.info())

# Guardar a CSV para importar en Power BI
df.to_csv('ventas_cosmeticos_limpio.csv', index=False)
print("\n¡Archivo 'ventas_cosmeticos_limpio.csv' guardado con éxito!")


--- Dataset Limpio y Preparado ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   ID_Venta         1500 non-null   object        
 1   Fecha            1500 non-null   datetime64[ns]
 2   Canal            1500 non-null   object        
 3   Categoria        1500 non-null   object        
 4   Producto         1500 non-null   object        
 5   Cantidad         1500 non-null   int32         
 6   Precio_Unitario  1500 non-null   float64       
 7   Costo_Unitario   1500 non-null   float64       
 8   Devuelto         1500 non-null   int64         
 9   Venta_Total      1500 non-null   float64       
 10  Costo_Total      1500 non-null   float64       
 11  Ganancia_Bruta   1500 non-null   float64       
 12  Margen_%         1500 non-null   float64       
 13  Venta_Neta       1500 non-null   float64       
dtypes: d

---
## Resumen de Pre-Análisis para Power BI

> **Notas clave pre-importación:**
> * El dataset resultante consta de **1,500 transacciones** limpias.
> * Se habilitaron campos clave (`Venta_Neta`, `Devuelto`, `Margen_%`) que servirán de base para las medidas **DAX** en Power BI.
> * **Siguiente paso:** Carga del archivo `ventas_cosmeticos_limpio.csv` en Power BI Desktop.